In [1]:
import json
import numpy as np
import pandas as pd
import random
from itertools import chain
from sentence_transformers import SentenceTransformer, util


random.seed(42)
np.random.seed(42)

# Load the CSV file
file_path = 'annotations.csv'  
df = pd.read_csv(file_path)
df = df.dropna()

data_as_list = df.values.tolist()

final_data = []
for data in data_as_list:
    if data[1] == 'W':
        gender = 'Female'
    else:
        gender = 'Male'
    final_data.append(["Post_text: "+data[2] + "\nResponse_text: "+data[3], data[4], gender])

formatted_data = np.array(final_data)

np.random.shuffle(formatted_data)

train_data = formatted_data[:6000]
ice_data = formatted_data[6000:12000]
test_data = formatted_data[12000:13200]



model = SentenceTransformer('bert-base-nli-mean-tokens')

def extract_questions(data):
    return [item[0] for item in data]

train_text = extract_questions(train_data)
in_context_text = extract_questions(ice_data)

train_embeddings = model.encode(train_text, convert_to_tensor=True)
in_context_embeddings = model.encode(in_context_text, convert_to_tensor=True)

similarity_matrix = util.cos_sim(train_embeddings, in_context_embeddings)

combinations = [
    ("2-sim-dissim", 2, {"similar": 1, "dissimilar": 1, "random": 0}),
    ("2-half-random", 2, {"similar": 1, "dissimilar": 0, "random": 1}),
    ("2-random", 2, {"similar": 0, "dissimilar": 0, "random": 2}),
    ("3-sim-dissim", 3, {"similar": 2, "dissimilar": 1, "random": 0}),
    ("3-half-random", 3, {"similar": 2, "dissimilar": 0, "random": 1}),
    ("3-random", 3, {"similar": 0, "dissimilar": 0, "random": 3}),
    ("4-sim-dissim", 4, {"similar": 2, "dissimilar": 2, "random": 0}),
    ("4-half-random", 4, {"similar": 2, "dissimilar": 0, "random": 2}),
    ("4-random", 4, {"similar": 0, "dissimilar": 0, "random": 4}),
    ("5-sim-dissim", 5, {"similar": 3, "dissimilar": 2, "random": 0}),
    ("5-half-random", 5, {"similar": 3, "dissimilar": 0, "random": 2}),
    ("5-random", 5, {"similar": 0, "dissimilar": 0, "random": 5}),
]

num_training = len(train_text)
groups = [list(range(num_training))[i:i + 500] for i in range(0, num_training, 500)]

final_indices = []
combination_types = []


for group, (comb_type, num_examples, counts) in zip(groups, combinations):
    for idx in group:
        similarities = similarity_matrix[idx]
        sorted_indices = similarities.argsort(descending=True)
        most_similar = sorted_indices[:counts["similar"]].tolist()
        most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
        random_indices = random.sample(range(len(in_context_text)), counts["random"]) if counts["random"] > 0 else []
        selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
        random.shuffle(selected_indices)
        final_indices.append(selected_indices)
        combination_types.append(comb_type)

merged_list_train = list(zip(final_indices, combination_types))



# Function to process test data for a dataset
def process_test_data(test_data, in_context_data, similarity_matrix, combinations):
    num_test_examples = len(test_data)
    group_size = num_test_examples // len(combinations)  # 22 examples per combination
    print(group_size)
    groups = [list(range(num_test_examples))[i:i + group_size] for i in range(0, num_test_examples, group_size)]

    final_indices = []
    combination_types = []

    for group, (comb_type, num_examples, counts) in zip(groups, combinations):
        for idx in group:
            similarities = similarity_matrix[idx]
            sorted_indices = similarities.argsort(descending=True)
            most_similar = sorted_indices[:counts["similar"]].tolist()
            most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
            random_indices = random.sample(range(len(in_context_data)), counts["random"]) if counts["random"] > 0 else []
            selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
            random.shuffle(selected_indices)
            final_indices.append(selected_indices)
            combination_types.append(comb_type)

    return final_indices, combination_types

test_text = extract_questions(test_data)
test_embeddings = model.encode(test_text, convert_to_tensor=True)
similarity_matrix_test = util.cos_sim(test_embeddings, in_context_embeddings)

final_indices_test, combination_types_test = process_test_data(
    test_text, in_context_text, similarity_matrix_test, combinations
)

# Merged list for test data in both datasets
merged_list_test = list(zip(final_indices_test, combination_types_test))



import numpy as np

import warnings
warnings.filterwarnings('ignore')



base_template = """Instruction:
You are an expert assistant trained to jointly predict the sentiment and the gender for the given input from social media post and its response. 

Possible types of sentiment are: 'Mixed', 'Negative', 'Neutral', and 'Positive'.  
Possible types of gender are: 'Male' and 'Female'.  

Output Format:
The output should be in the format: 'sentiment, gender'.

Examples:
"""

def put_example(data):
    
    template = """
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Answer: {sent}, {gend}
"""
    result_string = template.format(post=data[0], sent=data[1], gend=data[2])
    
    return result_string

def put_example_test(data):
    
    template = """
Now, solve for this example:
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Model Answer: {sent}, {gend}
"""
    result_string = template.format(post=data[0], sent=data[1], gend=data[2])
    
    return result_string

def put_example_test_for_test_data(data):
    template = """
Now, solve for this example:
{post}
Q: Predict the sentiment and the gender of the above post and response in the format sentiment, gender.
Model Answer: """
    result_string = template.format(post=data[0])
    
    return result_string





import ast
from datasets import Dataset, DatasetDict
def get_prompt(idx, data, Test=0):
    result = []
    temp = []
    target = []
    for index, id in enumerate(idx):
        temp_template = base_template
        indices_list = id[0]
        for i in range(len(indices_list)):
           example = put_example(ice_data[indices_list[i]])
           temp_template = temp_template + example
        
        if Test == 0:
            temp_template = temp_template + put_example_test(data[index])
        else:
            temp_template = temp_template + put_example_test_for_test_data(data[index])
        temp.append(temp_template)
        
        target.append((f'{data[index][1]}, {data[index][2]}'))
        result.append(f"{id[1]}")

    return {'input_text': temp, 'target_text': target, 'combination': result}
            
            
            
train_data = get_prompt(merged_list_train, train_data)
test_data = get_prompt(merged_list_test, test_data, Test=1)


train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)


#Convert DataFrames to Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

100


In [2]:
from peft import LoraConfig, PeftConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub.hf_api import HfFolder
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)



HfFolder.save_token(os.environ.get("HF_TOKEN"))
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"": "cuda:0"}, trust_remote_code=True,)

model.config.use_cache = True # silence the warnings
# model.config.pretraining_tp = 1
# model.gradient_checkpointing_enable()
# model.enable_input_require_grads()
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
#model.resize_token_embeddings(len(tokenizer))
#tokenizer.add_tokens(new_tokens)
#model.resize_token_embeddings(len(tokenizer))


model = PeftModel.from_pretrained(model, 'fine_tuned_llama3_rtgen_10epoch')


tokenizer.padding_side = "left"
tokenizer.pad_token_id = tokenizer.bos_token_id
tokenizer.pad_token = tokenizer.bos_token
model.config.pad_token_id = tokenizer.bos_token_id
model = model.bfloat16()
#model.resize_token_embeddings(len(tokenizer))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
import torch
from tqdm import tqdm  # Import tqdm for progress bar

# Ensure you're in evaluation mode
model.eval()

# Define the batch size
batch_size = 4  # Adjust based on your GPU memory

def infer_batch(model, tokenizer, input_texts, batch_size, max_input_length):
    predictions = []
    num_batches = (len(input_texts) + batch_size - 1) // batch_size  # Calculate number of batches

    with torch.no_grad():
        for i in tqdm(range(num_batches), desc="Processing Batches", unit="batch"):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, len(input_texts))
            batch = input_texts[start_idx:end_idx]
            
            # Tokenize the batch
            inputs = tokenizer(batch, return_tensors="pt", padding="max_length", truncation=True, max_length=max_input_length)
            #print(inputs['input_ids'])
            input_ids = inputs["input_ids"].to(model.device)
            attention_mask = inputs["attention_mask"].to(model.device)
            
            # Forward pass
            outputs = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=16, num_return_sequences=1)
            #prob = model(inputs[0])
            #print(outputs.tolist())
            # Decode the predictions
            batch_predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            for pred in batch_predictions:
                if 'Model ' in pred:
                    #print(pred)
                    _, result = pred.split('Model Answer:', 1)  # Split and get part after the delimiter
                    predictions.append(result.strip())
                else:
                    predictions.append(pred.strip())
    
    return predictions

# Test dataset
test_data = test_dataset['input_text']
# Get predictions
predictions = infer_batch(model, tokenizer, test_data, batch_size, max_input_length=1024)
#np.save('custom_loss_pred.npy', np.array(predictions))
# Optionally, compare with the true labels if needed
test_labels = test_dataset['target_text']


Processing Batches:   0%|          | 0/300 [00:00<?, ?batch/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
2024-12-06 10:49:18.276369: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-06 10:49:27.602707: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-06 10:49:27.602763: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-06 10:49:29.173620: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register

In [5]:
cleaned_res = [item.split('\n')[0] for item in predictions]
split_data_generated = [item.split(', ') for item in cleaned_res]
split_data_true = [item.split(', ') for item in test_labels]


In [6]:
gc = 0


gen_gend = []
true_gend = []
temp = []

sentiment_to_match = ['Positive', 'Negative', 'Neutral', 'Mixed']
gend_to_match = ['Male', 'Female']
for i, txt in enumerate(predictions):
    flag = 0
    for gend in gend_to_match:
        if gend.lower() in txt.lower() and flag == 0:
            #print(gend, split_data_true[i][1])
            if(gend.lower()==split_data_true[i][1].lower() or (gend.lower()=='fem' and split_data_true[i][1] == 'Female') or (gend.lower()=='feale' and split_data_true[i][1] == 'Female')):
                gc+=1
                gen_gend.append(split_data_true[i][1])
                temp.append(gend)
                true_gend.append(split_data_true[i][1])
                flag = 1
                break
    if flag == 0: 
        
        gen_gend.append('Male' if split_data_true[i][1] == 'Female' else 'Female')
        true_gend.append(split_data_true[i][1])
print(gc/len(predictions))
    

0.8666666666666667


In [7]:
from sentence_transformers import SentenceTransformer, util
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

sentiment_to_match = ['Positive', 'Negative', 'Neutral', 'Mixed']
profession_correct = 0
#gender_correct = 0
c = 0
true_sent = []
gen_sent = []

i = 0

for gen, true in zip(split_data_generated, split_data_true):
    print(i)
    i+=1
    flag = 0
    true_sent.append(true[0])
    #true_gend.append(true[1])
    temp = 'NA'
    if len(gen) < 2: 
        if (isinstance(gen, list)): gen = gen[0]
        gen_sentiment = gen
        true_sentiment, _ = true
        #print(gen, true)
        c+=1
        # gen_prof.append('NA')
        # if true[1] == 'Male':
        #     gen_gend.append('Female')
        # else:
        #     gen_gend.append('Male')
    elif(len(gen) > 2):
        #print(gen)
        gen = gen[:2]
        
        gen_sentiment, _ = gen
        true_sentiment, _ = true
    else: 
        gen_sentiment, _ = gen
        true_sentiment, _ = true
    
    #print(true_gender, gen_gender)
    # Profession check
    if true_sentiment.lower() in gen_sentiment.lower():
        gen_sent.append(true_sentiment)
        profession_correct += 1
        flag = 1
    elif(len(gen_sentiment)!= 0):
        print(true_sentiment,'|', gen_sentiment)
        
        if gen_sentiment not in sentiment_to_match:
            similarity_array=[]
            for sentiment in sentiment_to_match:
                word1=gen_sentiment
                word2=sentiment
                embeddings1 = bert_model.encode(word1, convert_to_tensor=True)
                embeddings2 = bert_model.encode(word2, convert_to_tensor=True)
                similarity_array.append((util.cos_sim(embeddings1, embeddings2).item(),sentiment))
            sorted_similarity_array = sorted(similarity_array, key=lambda x: x[0], reverse=True)
            temp = sorted_similarity_array[0][1]
            #print(sorted_similarity_array[0][1])
            if sorted_similarity_array[0][1] == true_sentiment.lower():
                #print(sorted_similarity_array[0][1], gen_profession)
                gen_sent.append(true_sentiment)
                profession_correct += 1
                flag = 1
        else:
            temp = gen_sentiment

    if flag == 0:
        #print(true_profession, gen_profession)
        gen_sent.append(temp)
        #print(true_profession,'|', gen_profession, '|', temp)

    # Gender check
    # if ': Male' in gen_gender or ': Fem' in gen_gender:
    #     gen_gender = gen_gender.split(': ')[1]
    #     #print(gen_gender)
    # if gen_gender == true_gender or (gen_gender == 'Fem' and true_gender == 'Female'):
    #     gender_correct += 1
    #     #print(gen_gender, true_gender)
    #     gen_gend.append(true_gender)
        
    # else:
    #     #print(gen_gender, true_gender)
    #     if true[1] == 'Male':
    #         gen_gend.append('Female')
    #     else:
    #         gen_gend.append('Male')


# Total number of samples
total_samples = len(split_data_true)

# Calculate accuracy
profession_accuracy = profession_correct / total_samples
#gender_accuracy = gender_correct / total_samples

print(f"Profession Accuracy: {profession_accuracy:.5f}")
#print(f"Gender Accuracy: {gender_accuracy:.5f}")

0
Neutral | Negative
1
2
3
4
Neutral | Positive
5
6
7
Negative | Positive
8
9
10
11
12
13
14
Negative | Positive
15
16
17
18
19
Positive | Neutral
20
21
22
Mixed | Negative
23
Neutral | Positive
24
25
26
27
28
29
30
Negative | Neutral
31
32
Negative | Neutral
33
Positive | Negative
34
Mixed | Positive
35
36
37
38
39
Neutral | Positive
40
41
Positive | Neutral
42
43
44
45
46
Positive | Negative
47
48
49
Neutral | Negative
50
51
52
Neutral | Positive
53
Neutral | Negative
54
55
Neutral | Positive
56
57
58
59
60
61
62
63
Neutral | Positive
64
Neutral | Negative
65
66
67
Neutral | Positive
68
Mixed | Positive
69
70
71
72
Mixed | Neutral
73
74
75
Negative | Mixed
76
77
78
79
80
81
Negative | Neutral
82
83
84
85
86
Neutral | Mixed
87
Positive | Neutral
88
89
Mixed | Positive
90
91
Neutral | Negative
92
93
94
95
96
97
98
Positive | Neutral
99
100
101
102
103
104
Positive | Neutral
105
Neutral | Positive
106
Mixed | Negative
107
Neutral | Positive
108
109
110
Mixed | Positive
111
112
Positive 